# QDQ Model Analysis — Stable Diffusion on CPU EP (Intel)

Analyzing the impact of QDQ quantization on CPU EP performance using Stable Diffusion component benchmarks.

**Data source**: `cpu-perf-sd-intel/results.json` — latency and op-level analysis for float, QDQ, and QDQ-with-cleanup variants of text_encoder, unet, vae_decoder, vae_encoder.

In [1]:
import json
import pandas as pd
from pathlib import Path

# Load results
results_path = Path(r"C:\Users\jambaykinley\OneDrive - Microsoft\Desktop\work-ARM\cpu-perf-sd-intel\results.json")
with open(results_path) as f:
    data = json.load(f)

latency = data["latency_ms"]
op_analysis = data["op_analysis"]
metadata = data["metadata"]

variants = ["float", "qdq", "qdq_cleanup"]
components = ["text_encoder", "unet", "vae_decoder", "vae_encoder"]

print(f"ORT version: {metadata['ort_version']}")
print(f"Platform: {metadata['platform']}")
print(f"Iterations: {metadata['iterations']}, Warmup: {metadata['warmup']}")

ORT version: 1.24.1
Platform: Windows-11-10.0.26200-SP0
Iterations: 10, Warmup: 3


## 1. Per-Component Latency Comparison

Mean latency (ms) for each SD component across float, QDQ, and QDQ-with-cleanup variants. Ratios show QDQ slowdown relative to float baseline.

In [2]:
# Build latency comparison table
rows = []
for comp in components:
    float_ms = latency["float"][comp]["mean_ms"]
    qdq_ms = latency["qdq"][comp]["mean_ms"]
    cleanup_ms = latency["qdq_cleanup"][comp]["mean_ms"]
    rows.append({
        "Component": comp,
        "Float (ms)": round(float_ms, 1),
        "QDQ (ms)": round(qdq_ms, 1),
        "QDQ Cleanup (ms)": round(cleanup_ms, 1),
        "QDQ / Float": f"{qdq_ms / float_ms:.2f}×",
        "Cleanup / Float": f"{cleanup_ms / float_ms:.2f}×",
        "Cleanup Improvement": f"{(1 - cleanup_ms / qdq_ms) * 100:.1f}%",
    })

df_latency = pd.DataFrame(rows)
df_latency

,Component,Float (ms),QDQ (ms),QDQ Cleanup (ms),QDQ / Float,Cleanup / Float,Cleanup Improvement
0,text_encoder,57.1,94.2,93.5,1.65×,1.64×,0.7%
1,unet,1977.9,4428.6,3819.6,2.24×,1.93×,13.8%
2,vae_decoder,5702.6,10349.9,7485.4,1.81×,1.31×,27.7%
3,vae_encoder,3024.1,4401.3,3855.9,1.46×,1.28×,12.4%


## 2. Op Count & Fusion Analysis

Total ops, Q/DQ counts, and key fused operators per component and variant. Shows how QDQ inflates op count and blocks fusions, and how cleanup partially restores them.

In [6]:
# Op count and fusion analysis
fused_ops = ["FusedMatMul", "QuickGelu"]

rows = []
for comp in components:
    for variant in variants:
        info = op_analysis[variant][comp]
        op_counts = info["op_counts"]
        row = {
            "Component": comp,
            "Variant": variant,
            "Total Ops": info["total_ops"],
            "Q": info["total_q"],
            "DQ": info["total_dq"],
        }
        for op in fused_ops:
            row[op] = op_counts.get(op, 0)
        rows.append(row)

df_ops = pd.DataFrame(rows)
df_ops

,Component,Variant,Total Ops,Q,DQ,FusedMatMul,QuickGelu
0,text_encoder,float,565,0,0,12,12
1,text_encoder,qdq,1097,206,487,0,12
2,text_encoder,qdq_cleanup,902,121,378,0,12
3,unet,float,2605,0,0,0,47
4,unet,qdq,4507,930,2050,0,0
5,unet,qdq_cleanup,3199,402,1355,0,47
6,vae_decoder,float,402,0,0,3,29
7,vae_decoder,qdq,958,212,460,0,0
8,vae_decoder,qdq_cleanup,594,66,285,0,29
9,vae_encoder,float,333,0,0,3,21


## 3. Q/DQ Breakdown — Activation vs Weight & Shape-Op Barriers

Shows how Q/DQ nodes break down into activation vs weight quantization, and how many are blocked from cleanup by intervening shape operators (Q → Reshape → DQ patterns).

In [5]:
# Q/DQ breakdown: activation vs weight, shape-op barriers
rows = []
for comp in components:
    for variant in ["qdq", "qdq_cleanup"]:
        info = op_analysis[variant][comp]
        rows.append({
            "Component": comp,
            "Variant": variant,
            "Q (activation)": info["q_activation"],
            "DQ (activation)": info["dq_activation"],
            "DQ (weight)": info["dq_weight"],
            "Q → shape op": info["q_consumed_by_shape_op"],
            "DQ ← shape op": info["dq_input_from_shape_op"],
            "Q shape %": f"{info['q_consumed_by_shape_op'] / max(info['q_activation'], 1) * 100:.0f}%",
        })

df_qdq = pd.DataFrame(rows)
df_qdq

,Component,Variant,Q (activation),DQ (activation),DQ (weight),Q → shape op,DQ ← shape op,Q shape %
0,text_encoder,qdq,206,256,231,121,145,59%
1,text_encoder,qdq_cleanup,121,147,231,121,145,100%
2,unet,qdq,930,1177,873,401,401,43%
3,unet,qdq_cleanup,402,482,873,401,401,100%
4,vae_decoder,qdq,212,258,202,66,68,31%
5,vae_decoder,qdq_cleanup,66,83,202,66,68,100%
6,vae_encoder,qdq,163,198,156,50,52,31%
7,vae_encoder,qdq_cleanup,51,65,156,50,52,98%


## 4. Key Observations

**Latency:**
- QDQ models are **1.5–2.2× slower** than float on CPU EP — the opposite of the expected speedup.
- `qdq_cleanup` helps (13–28% faster than raw QDQ for Conv-heavy components) but models **remain slower than float**.
- Cleanup is least effective for text_encoder (0.7% improvement) — most Q/DQ ops there pass through shape operators.

**Op Count & Fusions:**
- QDQ nearly **doubles** total op count (e.g. unet: 2,605 → 4,507).
- QuickGelu is **lost** in QDQ models (unet: 47 → 0). Cleanup restores it.
- FusedMatMul is **blocked** in QDQ text_encoder (12 → 0). Not restored by cleanup.

**Shape-Op Barriers to Cleanup:**
- A large fraction of activation Q ops flow into shape operators (unet: 401/930 = 43%, text_encoder: 121/206 = 59%).
- These Q → Reshape → DQ patterns are **not removed** by current `qdq_cleanup` — confirming the need to extend cleanup to handle them.
- **Weight DQ ops are never removed** (e.g. unet DQ(wt) stays at 873 in both QDQ and cleanup) — supports the suggestion to fold weight DQ into float initializers.
- After cleanup, remaining activation Q ops exactly equal the shape-blocked count (text_encoder: 121 = 121, unet: 402 ≈ 401), meaning cleanup successfully removes all non-shape-blocked pairs.